# Polymarket Feature Ablation — RQ2 with Literature-Motivated Main Specification

## Research question
> Which types of Polymarket information are most useful?

This notebook deliberately tests the wider corrected Polymarket feature set **separately**.

That is where we test variables excluded from the headline Enhanced specification, including:
- cumulative probability change;
- support-volume surge;
- relative support volume;
- participation;
- probability × contract volume.

## Headline Enhanced specification

The main Enhanced specification used in the dissertation's primary modelling notebooks is:

- `PM_Prob_Lag1`
- `PM_Change_1D_Lag1`
- `PM_Prob_x_TimeToResolution`
- `PM_Log_Cumulative_Support_Notional_Lag1`
- `PM_Net_Support_Imbalance_Lag1`

Traditional ML also includes `Days_to_Resolution_Years`.

## Why this separation matters

The headline model is not chosen because it gave the best final-test result.

Instead:
- the main model represents distinct literature-motivated information channels;
- this RQ2 notebook investigates which individual feature types appear useful;
- exploratory/robustness results are reported transparently rather than used to retroactively redesign the main hypothesis test.


## Step 1 — Upload the Two Required Files

Upload:

1. `Final_Competitive_Markets_MODEL_READY_with_CORRECTED_PM_Trading_Features.xlsx`
2. `Market_Specific_Robustness_All_Models_Results.xlsx`

The market-specific results file must come from the **new corrected market-specific robustness notebook**.


## Step 2 — Install and Import Packages


In [ ]:
# ============================================================
# STEP 2: Install and Import Packages
# ============================================================

!pip -q install xgboost openpyxl statsmodels scikit-learn scipy

import ast
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from scipy.special import softmax

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit

from xgboost import XGBClassifier

from statsmodels.discrete.conditional_models import ConditionalLogit
import statsmodels.api as sm

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 200)

print("Packages imported successfully.")


## Step 3 — Load the Model-Ready Dataset and Previous Selected Parameters


In [ ]:
# ============================================================
# STEP 3: Load Corrected Dataset and New Selected Parameters
# ============================================================

import glob
import os

data_candidates = [
    x for x in glob.glob("/content/*.xlsx")
    if "CORRECTED_PM_Trading_Features" in os.path.basename(x)
    and "MODEL_READY" in os.path.basename(x).upper()
]

results_candidates = [
    x for x in glob.glob("/content/*.xlsx")
    if "Market_Specific_Robustness_All_Models_Results" in os.path.basename(x)
]

if len(data_candidates) != 1:
    raise ValueError(
        f"Could not uniquely detect the corrected dataset: {data_candidates}"
    )

if len(results_candidates) != 1:
    raise ValueError(
        f"Could not uniquely detect the NEW market-specific results: {results_candidates}"
    )

DATA_PATH = data_candidates[0]
RESULTS_PATH = results_candidates[0]

df = pd.read_excel(
    DATA_PATH,
    sheet_name="Model_Ready_Master"
)

best_ml = pd.read_excel(
    RESULTS_PATH,
    sheet_name="Best_ML_Params"
)

for col in [
    "Date",
    "Next_Financial_Date",
    "Financial_Prev1_Date",
    "Financial_Prev2_Date",
    "Financial_Prev3_Date",
    "PM_Signal_Date_Lag1",
    "PM_Refined_Activity_Signal_Date_Lag1"
]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col])

df = df.sort_values(
    ["Market_ID", "Date", "Event_ID", "Asset"]
).reset_index(drop=True)

print("Dataset:", DATA_PATH)
print("Results:", RESULTS_PATH)
print("Rows:", len(df))
print("Events:", df["Event_ID"].nunique())

print("\nEvents by market:")
print(
    df.groupby("Market_ID")["Event_ID"].nunique()
)

display(best_ml.head())


## Step 4 — Define the Expanded RQ2 Ablation Specifications

The Baseline remains unchanged.

The new specifications test probability information and corrected trading information separately, while keeping the same target, dates and event structure.


In [ ]:
# ============================================================
# STEP 4: Define Literature-Motivated RQ2 Ablation Specifications
# ============================================================

BASELINE = [
    "Lag1_Return",
    "Lag2_Return",
    "Lag3_Return"
]

MAIN_PM = [
    "PM_Prob_Lag1",
    "PM_Change_1D_Lag1",
    "PM_Prob_x_TimeToResolution",
    "PM_Log_Cumulative_Support_Notional_Lag1",
    "PM_Net_Support_Imbalance_Lag1"
]

CL_SPECS = {
    "Baseline":
        BASELINE,

    # -----------------------------
    # Original probability channels
    # -----------------------------
    "PM_Level":
        BASELINE + ["PM_Prob_Lag1"],

    "PM_Daily_Change":
        BASELINE + ["PM_Change_1D_Lag1"],

    # Kept for RQ2 robustness, not headline Enhanced
    "PM_Cumulative_Change":
        BASELINE + ["PM_Change_From_Start_Lag1"],

    "PM_Time_Interaction":
        BASELINE + ["PM_Prob_x_TimeToResolution"],

    # -----------------------------
    # Corrected trading channels
    # -----------------------------
    "Support_Volume":
        BASELINE + ["PM_Log_Support_Notional_Lag1"],

    "Relative_Support_Volume":
        BASELINE + ["PM_Relative_Support_Volume_Share_Lag1"],

    "Cumulative_Support_Volume":
        BASELINE + ["PM_Log_Cumulative_Support_Notional_Lag1"],

    "Relative_Cumulative_Support":
        BASELINE + ["PM_Relative_Cumulative_Support_Share_Lag1"],

    "Support_Participation":
        BASELINE + ["PM_Log_Support_Unique_Takers_Lag1"],

    "Net_Support_Imbalance":
        BASELINE + ["PM_Net_Support_Imbalance_Lag1"],

    # Kept for RQ2 robustness, not headline Enhanced
    "Support_Volume_Surge":
        BASELINE + ["PM_Support_Volume_Surge_Lag1"],

    "PM_x_Contract_Volume":
        BASELINE
        + [
            "PM_Prob_Lag1",
            "PM_Prob_x_Log_Contract_Notional_Lag1"
        ],

    # This now EXACTLY matches the headline Conditional Logit specification.
    "Full_Enhanced":
        BASELINE + MAIN_PM
}

ML_SPECS = {
    name: (
        features
        if name != "Full_Enhanced"
        else features + ["Days_to_Resolution_Years"]
    )
    for name, features in CL_SPECS.items()
}

MARKETS = [
    "BTC_GOLD_2025",
    "BTC_GOLD_2026",
    "BTC_GOLD_SP500"
]

ML_MODELS = [
    "Logistic_Regression",
    "Random_Forest",
    "XGBoost"
]

ALL_MODELS = [
    "Conditional_Logit",
    "Logistic_Regression",
    "Random_Forest",
    "XGBoost"
]

feature_set_rows = []

for spec_name, features in CL_SPECS.items():
    feature_set_rows.append({
        "Specification": spec_name,
        "Model_Family": "Conditional Logit",
        "Features": " + ".join(features)
    })

for spec_name, features in ML_SPECS.items():
    feature_set_rows.append({
        "Specification": spec_name,
        "Model_Family": "Traditional ML",
        "Features": " + ".join(features)
    })

feature_set_table = pd.DataFrame(feature_set_rows)

required_features = sorted({
    feature
    for spec in ML_SPECS.values()
    for feature in spec
})

assert df[required_features].notna().all().all(), (
    "A main ablation feature contains missing values."
)

print("HEADLINE MAIN PM FEATURES:")
print(MAIN_PM)

print("\nConditional Logit specifications:")
for k, v in CL_SPECS.items():
    print(k, "->", v)

print("\nTraditional ML specifications:")
for k, v in ML_SPECS.items():
    print(k, "->", v)


## Step 5 — Basic No-Leak and Structure Checks


In [ ]:
# ============================================================
# STEP 5: No-Leak and Event Checks
# ============================================================

assert (
    df["PM_Signal_Date_Lag1"]
    < df["Date"]
).all(), "Polymarket lag date leakage detected."

assert (
    df["PM_Refined_Activity_Signal_Date_Lag1"]
    < df["Date"]
).all(), "Corrected PM activity timing leakage detected."

assert (
    df["Financial_Prev1_Date"]
    < df["Date"]
).all(), "Lag1 financial date leakage detected."

assert (
    df["Date"]
    < df["Next_Financial_Date"]
).all(), "Forward target timing problem detected."

winner_count = (
    df.groupby("Event_ID")["Winner"]
    .sum()
)

assert (
    winner_count == 1
).all(), "Each Event_ID must have exactly one winner."

print("All timing and event-structure checks passed.")


## Step 6 — Recreate the Same Market-Specific Chronological Splits

For each market:

- first ~80% of dates = development;
- last ~20% = untouched market-specific final holdout;
- four expanding-window validation folds inside development.


In [ ]:
# ============================================================
# STEP 6A: Market-Specific 80/20 Splits
# ============================================================

market_splits = {}

for market_id in MARKETS:

    market_df = df[
        df["Market_ID"] == market_id
    ].copy()

    dates = np.array(
        sorted(market_df["Date"].unique())
    )

    cut = int(
        np.floor(
            0.80 * len(dates)
        )
    )

    market_splits[
        market_id
    ] = {
        "Dev_Dates":
            dates[:cut],
        "Test_Dates":
            dates[cut:]
    }

    print(
        market_id,
        "| Total:",
        len(dates),
        "| Development:",
        cut,
        "| Final holdout:",
        len(dates) - cut
    )


In [ ]:
# ============================================================
# STEP 6B: Same Four Expanding-Window Folds
# ============================================================

def build_market_folds(
    dev_dates,
    n_splits=4
):

    n = len(dev_dates)

    test_size = max(
        8,
        int(
            np.floor(
                0.10 * n
            )
        )
    )

    while (
        n_splits * test_size
        >= int(0.60 * n)
    ):
        test_size -= 1

    if test_size < 5:
        raise ValueError(
            "Insufficient development dates."
        )

    splitter = TimeSeriesSplit(
        n_splits=n_splits,
        test_size=test_size,
        gap=0
    )

    folds = []

    for fold_no, (
        train_idx,
        val_idx
    ) in enumerate(
        splitter.split(dev_dates),
        start=1
    ):

        folds.append({
            "Fold":
                fold_no,
            "Train_Dates":
                dev_dates[train_idx],
            "Val_Dates":
                dev_dates[val_idx]
        })

    return folds


market_folds = {}

for market_id in MARKETS:

    market_folds[
        market_id
    ] = build_market_folds(
        market_splits[
            market_id
        ]["Dev_Dates"]
    )

    print("\n", market_id)

    for fold in market_folds[
        market_id
    ]:
        print(
            "Fold",
            fold["Fold"],
            ":",
            pd.Timestamp(
                fold[
                    "Val_Dates"
                ][0]
            ).date(),
            "to",
            pd.Timestamp(
                fold[
                    "Val_Dates"
                ][-1]
            ).date()
        )


## Step 7 — Freeze the Previously Selected Baseline ML Hyperparameters

This is important.

For each market and model, we take the hyperparameters selected previously for the **Baseline** model and use exactly those same parameters for:

- Baseline
- PM Level
- PM Daily Change
- PM Cumulative Change
- PM × Time
- Full Enhanced

Therefore we are testing the **feature contribution**, not giving each feature version another opportunity to tune itself.


In [ ]:
# ============================================================
# STEP 7: Parse and Freeze Baseline Hyperparameters
# ============================================================

baseline_ml_params = {}

baseline_param_rows = best_ml[
    best_ml[
        "Specification"
    ] == "Baseline"
].copy()

for _, row in baseline_param_rows.iterrows():

    key = (
        row["Market_ID"],
        row["Model"]
    )

    baseline_ml_params[
        key
    ] = ast.literal_eval(
        row["Parameters"]
    )

expected = (
    len(MARKETS)
    * len(ML_MODELS)
)

print(
    "Frozen market-model parameter sets:",
    len(baseline_ml_params)
)

assert (
    len(baseline_ml_params)
    == expected
), (
    "Expected one Baseline parameter set "
    "for every market-model combination."
)

for key, value in baseline_ml_params.items():
    print(key, "->", value)


## Step 8 — Event-Level Evaluation Function


In [ ]:
# ============================================================
# STEP 8: Event-Level Metrics
# ============================================================

def evaluate_event_predictions(
    pred
):

    eps = 1e-15
    rows = []

    for event_id, g in pred.groupby(
        "Event_ID"
    ):

        g = g.copy()

        probs = g[
            "Pred_Prob"
        ].to_numpy(
            dtype=float
        )

        probs = probs / probs.sum()

        y = g[
            "Winner"
        ].to_numpy(
            dtype=int
        )

        actual_idx = np.where(
            y == 1
        )[0][0]

        predicted_idx = int(
            np.argmax(
                probs
            )
        )

        actual_winner = g.iloc[
            actual_idx
        ]["Asset"]

        predicted_winner = g.iloc[
            predicted_idx
        ]["Asset"]

        winner_prob = float(
            np.clip(
                probs[
                    actual_idx
                ],
                eps,
                1 - eps
            )
        )

        rows.append({
            "Market_ID":
                g[
                    "Market_ID"
                ].iloc[0],
            "Event_ID":
                event_id,
            "Date":
                g[
                    "Date"
                ].iloc[0],
            "Actual_Winner":
                actual_winner,
            "Predicted_Winner":
                predicted_winner,
            "Winner_Probability":
                winner_prob,
            "Correct":
                int(
                    actual_winner
                    == predicted_winner
                ),
            "Log_Loss":
                -np.log(
                    winner_prob
                ),
            "Brier_Score":
                np.sum(
                    (
                        probs - y
                    ) ** 2
                )
        })

    return pd.DataFrame(
        rows
    )


## Step 9 — Conditional Logit Helper


In [ ]:
# ============================================================
# STEP 9: Conditional Logit Fit and Predict
# ============================================================

def fit_clogit_predict(
    train_df,
    test_df,
    features
):

    train_df = (
        train_df
        .reset_index(
            drop=True
        )
        .copy()
    )

    test_df = (
        test_df
        .reset_index(
            drop=True
        )
        .copy()
    )

    scaler = StandardScaler()

    X_train = scaler.fit_transform(
        train_df[
            features
        ]
    )

    X_test = scaler.transform(
        test_df[
            features
        ]
    )

    y_train = train_df[
        "Winner"
    ].astype(
        int
    ).to_numpy()

    groups = train_df[
        "Event_ID"
    ].astype(
        str
    ).to_numpy()

    model = ConditionalLogit(
        y_train,
        X_train,
        groups=groups
    )

    result = model.fit(
        method="bfgs",
        maxiter=1000,
        disp=False
    )

    pred = test_df[
        [
            "Market_ID",
            "Event_ID",
            "Date",
            "Asset",
            "Winner"
        ]
    ].copy()

    pred[
        "Utility"
    ] = (
        X_test
        @ np.asarray(
            result.params
        )
    )

    pred[
        "Pred_Prob"
    ] = (
        pred
        .groupby(
            "Event_ID"
        )[
            "Utility"
        ]
        .transform(
            lambda x:
                softmax(
                    x.to_numpy()
                )
        )
    )

    return pred


## Step 10 — Traditional ML Helpers


In [ ]:
# ============================================================
# STEP 10A: Build Frozen Traditional ML Model
# ============================================================

def build_model(
    model_name,
    params
):

    if (
        model_name
        == "Logistic_Regression"
    ):

        return Pipeline([
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                LogisticRegression(
                    C=params["C"],
                    penalty="l2",
                    solver="lbfgs",
                    max_iter=2000,
                    random_state=42
                )
            )
        ])

    if (
        model_name
        == "Random_Forest"
    ):

        return RandomForestClassifier(
            n_estimators=
                params[
                    "n_estimators"
                ],
            max_depth=
                params[
                    "max_depth"
                ],
            min_samples_leaf=
                params[
                    "min_samples_leaf"
                ],
            max_features=
                params[
                    "max_features"
                ],
            random_state=42,
            n_jobs=-1
        )

    if (
        model_name
        == "XGBoost"
    ):

        return XGBClassifier(
            n_estimators=
                params[
                    "n_estimators"
                ],
            max_depth=
                params[
                    "max_depth"
                ],
            learning_rate=
                params[
                    "learning_rate"
                ],
            subsample=
                params[
                    "subsample"
                ],
            colsample_bytree=
                params[
                    "colsample_bytree"
                ],
            objective=
                "binary:logistic",
            eval_metric=
                "logloss",
            random_state=42,
            n_jobs=-1,
            tree_method="hist",
            verbosity=0
        )

    raise ValueError(
        "Unknown model."
    )


In [ ]:
# ============================================================
# STEP 10B: Convert ML Scores to Competitive Probabilities
# ============================================================

def ml_event_predictions(
    model,
    test_df,
    features
):

    test_df = (
        test_df
        .reset_index(
            drop=True
        )
        .copy()
    )

    raw_prob = model.predict_proba(
        test_df[
            features
        ]
    )[:, 1]

    pred = test_df[
        [
            "Market_ID",
            "Event_ID",
            "Date",
            "Asset",
            "Winner"
        ]
    ].copy()

    pred[
        "Raw_Winner_Prob"
    ] = raw_prob

    pred[
        "Pred_Prob"
    ] = np.nan

    for event_id, idx in pred.groupby(
        "Event_ID"
    ).groups.items():

        idx = list(
            idx
        )

        probs = pred.loc[
            idx,
            "Raw_Winner_Prob"
        ].to_numpy(
            dtype=float
        )

        probs = np.clip(
            probs,
            1e-15,
            None
        )

        pred.loc[
            idx,
            "Pred_Prob"
        ] = (
            probs
            / probs.sum()
        )

    return pred


## Step 11 — Run Conditional Logit Ablation: Expanding-Window Validation


In [ ]:
# ============================================================
# STEP 11: Conditional Logit CV Ablation
# ============================================================

cv_event_tables = []

for market_id in MARKETS:

    market_df = df[
        df[
            "Market_ID"
        ] == market_id
    ].copy()

    print(
        "\nConditional Logit CV:",
        market_id
    )

    for spec_name, features in CL_SPECS.items():

        for fold in market_folds[
            market_id
        ]:

            train_df = market_df[
                market_df[
                    "Date"
                ].isin(
                    fold[
                        "Train_Dates"
                    ]
                )
            ].copy()

            val_df = market_df[
                market_df[
                    "Date"
                ].isin(
                    fold[
                        "Val_Dates"
                    ]
                )
            ].copy()

            pred = fit_clogit_predict(
                train_df,
                val_df,
                features
            )

            events = (
                evaluate_event_predictions(
                    pred
                )
            )

            events[
                "Model"
            ] = "Conditional_Logit"

            events[
                "Specification"
            ] = spec_name

            events[
                "OOS_Source"
            ] = "Expanding_Validation"

            events[
                "Fold"
            ] = fold["Fold"]

            events[
                "OOS_Block"
            ] = (
                "Fold_"
                + str(
                    fold[
                        "Fold"
                    ]
                )
            )

            cv_event_tables.append(
                events
            )

print(
    "Conditional Logit CV completed."
)


## Step 12 — Run Traditional ML Ablation: Expanding-Window Validation

The same frozen Baseline hyperparameters are used across all specifications.


In [ ]:
# ============================================================
# STEP 12: Traditional ML CV Ablation
# ============================================================

for market_id in MARKETS:

    market_df = df[
        df[
            "Market_ID"
        ] == market_id
    ].copy()

    print(
        "\nTraditional ML CV:",
        market_id
    )

    for model_name in ML_MODELS:

        params = (
            baseline_ml_params[
                (
                    market_id,
                    model_name
                )
            ]
        )

        for spec_name, features in ML_SPECS.items():

            for fold in market_folds[
                market_id
            ]:

                train_df = market_df[
                    market_df[
                        "Date"
                    ].isin(
                        fold[
                            "Train_Dates"
                        ]
                    )
                ].copy()

                val_df = market_df[
                    market_df[
                        "Date"
                    ].isin(
                        fold[
                            "Val_Dates"
                        ]
                    )
                ].copy()

                model = build_model(
                    model_name,
                    params
                )

                model.fit(
                    train_df[
                        features
                    ],
                    train_df[
                        "Winner"
                    ].astype(
                        int
                    )
                )

                pred = (
                    ml_event_predictions(
                        model,
                        val_df,
                        features
                    )
                )

                events = (
                    evaluate_event_predictions(
                        pred
                    )
                )

                events[
                    "Model"
                ] = model_name

                events[
                    "Specification"
                ] = spec_name

                events[
                    "OOS_Source"
                ] = "Expanding_Validation"

                events[
                    "Fold"
                ] = fold["Fold"]

                events[
                    "OOS_Block"
                ] = (
                    "Fold_"
                    + str(
                        fold[
                            "Fold"
                        ]
                    )
                )

                cv_event_tables.append(
                    events
                )

cv_events = pd.concat(
    cv_event_tables,
    ignore_index=True
)

print(
    "Total CV event-model-specification rows:",
    len(cv_events)
)


## Step 13 — Run All Ablation Specifications on Each Market's Final Holdout

The final holdout remains market-specific:

- BTC–Gold 2025 receives its own last 20%;
- BTC–Gold 2026 receives its own last 20%;
- BTC–Gold–S&P receives its own last 20%.


In [ ]:
# ============================================================
# STEP 13: Final Holdout Ablation
# ============================================================

final_event_tables = []

for market_id in MARKETS:

    market_df = df[
        df[
            "Market_ID"
        ] == market_id
    ].copy()

    dev_dates = (
        market_splits[
            market_id
        ][
            "Dev_Dates"
        ]
    )

    test_dates = (
        market_splits[
            market_id
        ][
            "Test_Dates"
        ]
    )

    train_df = market_df[
        market_df[
            "Date"
        ].isin(
            dev_dates
        )
    ].copy()

    test_df = market_df[
        market_df[
            "Date"
        ].isin(
            test_dates
        )
    ].copy()

    print(
        "\nFinal holdout:",
        market_id
    )

    # --------------------------------------------------------
    # Conditional Logit
    # --------------------------------------------------------

    for spec_name, features in CL_SPECS.items():

        pred = fit_clogit_predict(
            train_df,
            test_df,
            features
        )

        events = evaluate_event_predictions(
            pred
        )

        events[
            "Model"
        ] = "Conditional_Logit"

        events[
            "Specification"
        ] = spec_name

        events[
            "OOS_Source"
        ] = "Final_Holdout"

        events[
            "Fold"
        ] = np.nan

        events[
            "OOS_Block"
        ] = "Final_Holdout"

        final_event_tables.append(
            events
        )

    # --------------------------------------------------------
    # Traditional ML
    # --------------------------------------------------------

    for model_name in ML_MODELS:

        params = (
            baseline_ml_params[
                (
                    market_id,
                    model_name
                )
            ]
        )

        for spec_name, features in ML_SPECS.items():

            model = build_model(
                model_name,
                params
            )

            model.fit(
                train_df[
                    features
                ],
                train_df[
                    "Winner"
                ].astype(
                    int
                )
            )

            pred = ml_event_predictions(
                model,
                test_df,
                features
            )

            events = evaluate_event_predictions(
                pred
            )

            events[
                "Model"
            ] = model_name

            events[
                "Specification"
            ] = spec_name

            events[
                "OOS_Source"
            ] = "Final_Holdout"

            events[
                "Fold"
            ] = np.nan

            events[
                "OOS_Block"
            ] = "Final_Holdout"

            final_event_tables.append(
                events
            )

final_events = pd.concat(
    final_event_tables,
    ignore_index=True
)

print(
    "Final holdout event-model-specification rows:",
    len(final_events)
)


## Step 14 — Combine All Genuine Out-of-Sample Predictions


In [ ]:
# ============================================================
# STEP 14: Combine CV + Final Holdout OOS Predictions
# ============================================================

all_oos = pd.concat(
    [
        cv_events,
        final_events
    ],
    ignore_index=True
)

all_oos = all_oos.sort_values(
    [
        "Market_ID",
        "Date",
        "Model",
        "Specification"
    ]
).reset_index(
    drop=True
)

print(
    "All OOS rows:",
    len(all_oos)
)

print("\nOOS sources:")
print(
    all_oos[
        "OOS_Source"
    ].value_counts()
)


## Step 15 — Expanding-Window Validation Summary

Primary comparison: mean validation Log Loss.


In [ ]:
# ============================================================
# STEP 15: CV Summary
# ============================================================

cv_summary = (
    cv_events.groupby(
        [
            "Market_ID",
            "Model",
            "Specification"
        ]
    )
    .agg(
        CV_Events=(
            "Event_ID",
            "count"
        ),
        Mean_CV_Log_Loss=(
            "Log_Loss",
            "mean"
        ),
        Mean_CV_Brier=(
            "Brier_Score",
            "mean"
        ),
        Mean_CV_Accuracy=(
            "Correct",
            "mean"
        )
    )
    .reset_index()
)

display(
    cv_summary.sort_values(
        [
            "Market_ID",
            "Model",
            "Mean_CV_Log_Loss"
        ]
    )
)


## Step 16 — Final-Holdout Summary


In [ ]:
# ============================================================
# STEP 16: Final Holdout Summary
# ============================================================

final_summary = (
    final_events.groupby(
        [
            "Market_ID",
            "Model",
            "Specification"
        ]
    )
    .agg(
        Final_Events=(
            "Event_ID",
            "count"
        ),
        Final_Log_Loss=(
            "Log_Loss",
            "mean"
        ),
        Final_Brier=(
            "Brier_Score",
            "mean"
        ),
        Final_Accuracy=(
            "Correct",
            "mean"
        )
    )
    .reset_index()
)

display(
    final_summary.sort_values(
        [
            "Market_ID",
            "Model",
            "Final_Log_Loss"
        ]
    )
)


## Step 17 — Calculate Each Feature's Gain Relative to the Frozen Baseline

Positive Log-Loss Gain means the Polymarket specification improves on Baseline.


In [ ]:
# ============================================================
# STEP 17: Compare Each Specification with Baseline
# ============================================================

cv_base = (
    cv_summary[
        cv_summary[
            "Specification"
        ] == "Baseline"
    ][
        [
            "Market_ID",
            "Model",
            "Mean_CV_Log_Loss",
            "Mean_CV_Brier",
            "Mean_CV_Accuracy"
        ]
    ]
    .rename(
        columns={
            "Mean_CV_Log_Loss":
                "Baseline_CV_Log_Loss",
            "Mean_CV_Brier":
                "Baseline_CV_Brier",
            "Mean_CV_Accuracy":
                "Baseline_CV_Accuracy"
        }
    )
)

cv_gain = cv_summary.merge(
    cv_base,
    on=[
        "Market_ID",
        "Model"
    ],
    how="left"
)

cv_gain[
    "CV_LogLoss_Gain"
] = (
    cv_gain[
        "Baseline_CV_Log_Loss"
    ]
    -
    cv_gain[
        "Mean_CV_Log_Loss"
    ]
)

cv_gain[
    "CV_Brier_Gain"
] = (
    cv_gain[
        "Baseline_CV_Brier"
    ]
    -
    cv_gain[
        "Mean_CV_Brier"
    ]
)

cv_gain[
    "CV_Accuracy_Gain"
] = (
    cv_gain[
        "Mean_CV_Accuracy"
    ]
    -
    cv_gain[
        "Baseline_CV_Accuracy"
    ]
)


final_base = (
    final_summary[
        final_summary[
            "Specification"
        ] == "Baseline"
    ][
        [
            "Market_ID",
            "Model",
            "Final_Log_Loss",
            "Final_Brier",
            "Final_Accuracy"
        ]
    ]
    .rename(
        columns={
            "Final_Log_Loss":
                "Baseline_Final_Log_Loss",
            "Final_Brier":
                "Baseline_Final_Brier",
            "Final_Accuracy":
                "Baseline_Final_Accuracy"
        }
    )
)

final_gain = final_summary.merge(
    final_base,
    on=[
        "Market_ID",
        "Model"
    ],
    how="left"
)

final_gain[
    "Final_LogLoss_Gain"
] = (
    final_gain[
        "Baseline_Final_Log_Loss"
    ]
    -
    final_gain[
        "Final_Log_Loss"
    ]
)

final_gain[
    "Final_Brier_Gain"
] = (
    final_gain[
        "Baseline_Final_Brier"
    ]
    -
    final_gain[
        "Final_Brier"
    ]
)

final_gain[
    "Final_Accuracy_Gain"
] = (
    final_gain[
        "Final_Accuracy"
    ]
    -
    final_gain[
        "Baseline_Final_Accuracy"
    ]
)

print(
    "Positive LogLoss_Gain = Polymarket specification helped."
)


## Step 18 — Combine Validation and Final-Holdout Evidence

A simple robustness label is created:

- **Robust Improvement** = Log Loss improves in both CV and final holdout
- **Final Only** = improves only in final holdout
- **CV Only** = improves only in expanding validation
- **No Improvement** = neither improves


In [ ]:
# ============================================================
# STEP 18: Robustness Classification
# ============================================================

comparison = cv_gain.merge(
    final_gain[
        [
            "Market_ID",
            "Model",
            "Specification",
            "Final_LogLoss_Gain",
            "Final_Brier_Gain",
            "Final_Accuracy_Gain",
            "Final_Log_Loss",
            "Final_Brier",
            "Final_Accuracy"
        ]
    ],
    on=[
        "Market_ID",
        "Model",
        "Specification"
    ],
    how="inner"
)

def classify_result(row):

    if (
        row["Specification"]
        == "Baseline"
    ):
        return "Reference"

    cv_help = (
        row[
            "CV_LogLoss_Gain"
        ] > 0
    )

    final_help = (
        row[
            "Final_LogLoss_Gain"
        ] > 0
    )

    if cv_help and final_help:
        return "Robust Improvement"

    if final_help:
        return "Final Only"

    if cv_help:
        return "CV Only"

    return "No Improvement"


comparison[
    "Evidence_Label"
] = comparison.apply(
    classify_result,
    axis=1
)

display(
    comparison[
        [
            "Market_ID",
            "Model",
            "Specification",
            "CV_LogLoss_Gain",
            "Final_LogLoss_Gain",
            "Final_Accuracy_Gain",
            "Evidence_Label"
        ]
    ].sort_values(
        [
            "Market_ID",
            "Model",
            "Specification"
        ]
    )
)


## Step 19 — Feature Consistency Across Markets and Models

This is a descriptive cross-model summary.

For each Polymarket feature specification, count how many of the 12 market-model combinations show:

- improved CV Log Loss;
- improved final Log Loss;
- improvement in both.


In [ ]:
# ============================================================
# STEP 19: Feature Consistency Summary
# ============================================================

non_base = comparison[
    comparison[
        "Specification"
    ] != "Baseline"
].copy()

non_base[
    "CV_Improved"
] = (
    non_base[
        "CV_LogLoss_Gain"
    ] > 0
).astype(int)

non_base[
    "Final_Improved"
] = (
    non_base[
        "Final_LogLoss_Gain"
    ] > 0
).astype(int)

non_base[
    "Both_Improved"
] = (
    (
        non_base[
            "CV_Improved"
        ] == 1
    )
    &
    (
        non_base[
            "Final_Improved"
        ] == 1
    )
).astype(int)

feature_consistency = (
    non_base.groupby(
        "Specification"
    )
    .agg(
        Market_Model_Tests=(
            "Model",
            "count"
        ),
        CV_Improved_Count=(
            "CV_Improved",
            "sum"
        ),
        Final_Improved_Count=(
            "Final_Improved",
            "sum"
        ),
        Both_Improved_Count=(
            "Both_Improved",
            "sum"
        ),
        Mean_CV_LogLoss_Gain=(
            "CV_LogLoss_Gain",
            "mean"
        ),
        Mean_Final_LogLoss_Gain=(
            "Final_LogLoss_Gain",
            "mean"
        )
    )
    .reset_index()
)

feature_consistency[
    "Both_Improved_Share"
] = (
    feature_consistency[
        "Both_Improved_Count"
    ]
    /
    feature_consistency[
        "Market_Model_Tests"
    ]
)

display(
    feature_consistency.sort_values(
        [
            "Both_Improved_Count",
            "Mean_Final_LogLoss_Gain"
        ],
        ascending=[
            False,
            False
        ]
    )
)


## Step 20 — Market-Specific Feature Summary

This helps answer whether a feature works only for the near-resolution 2025 contract or also in the 2026 markets.


In [ ]:
# ============================================================
# STEP 20: Market-Specific Feature Summary
# ============================================================

market_feature_summary = (
    non_base.groupby(
        [
            "Market_ID",
            "Specification"
        ]
    )
    .agg(
        Models_Tested=(
            "Model",
            "count"
        ),
        Models_CV_Improved=(
            "CV_Improved",
            "sum"
        ),
        Models_Final_Improved=(
            "Final_Improved",
            "sum"
        ),
        Models_Both_Improved=(
            "Both_Improved",
            "sum"
        ),
        Mean_CV_LogLoss_Gain=(
            "CV_LogLoss_Gain",
            "mean"
        ),
        Mean_Final_LogLoss_Gain=(
            "Final_LogLoss_Gain",
            "mean"
        ),
        Mean_Final_Accuracy_Gain=(
            "Final_Accuracy_Gain",
            "mean"
        )
    )
    .reset_index()
)

display(
    market_feature_summary.sort_values(
        [
            "Market_ID",
            "Models_Both_Improved",
            "Mean_Final_LogLoss_Gain"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
)


## Step 21 — Event-Level Baseline vs Feature Statistical Comparison

For each market, model and Polymarket specification, we compare event-level out-of-sample Log Loss with its corresponding Baseline.

A Newey–West HAC regression is used on the time-ordered loss difference.

Interpretation:

`Loss_Difference = Baseline Log Loss - Feature Log Loss`

- Positive mean = Polymarket feature is better
- Negative mean = Baseline is better


In [ ]:
# ============================================================
# STEP 21: HAC Tests on OOS Event Log-Loss Differences
# ============================================================

hac_rows = []

baseline_events = (
    all_oos[
        all_oos[
            "Specification"
        ] == "Baseline"
    ][
        [
            "Market_ID",
            "Model",
            "Event_ID",
            "Date",
            "OOS_Block",
            "Log_Loss"
        ]
    ]
    .rename(
        columns={
            "Log_Loss":
                "Baseline_Log_Loss"
        }
    )
)

for spec_name in [x for x in CL_SPECS.keys() if x != "Baseline"]:

    feature_events = (
        all_oos[
            all_oos[
                "Specification"
            ] == spec_name
        ][
            [
                "Market_ID",
                "Model",
                "Event_ID",
                "Date",
                "OOS_Block",
                "Log_Loss"
            ]
        ]
        .rename(
            columns={
                "Log_Loss":
                    "Feature_Log_Loss"
            }
        )
    )

    paired_loss = baseline_events.merge(
        feature_events,
        on=[
            "Market_ID",
            "Model",
            "Event_ID",
            "Date",
            "OOS_Block"
        ],
        how="inner",
        validate="one_to_one"
    )

    paired_loss[
        "Loss_Difference"
    ] = (
        paired_loss[
            "Baseline_Log_Loss"
        ]
        -
        paired_loss[
            "Feature_Log_Loss"
        ]
    )

    for (
        market_id,
        model_name
    ), g in paired_loss.groupby(
        [
            "Market_ID",
            "Model"
        ]
    ):

        g = g.sort_values(
            "Date"
        ).copy()

        y = g[
            "Loss_Difference"
        ].to_numpy(
            dtype=float
        )

        X = np.ones(
            (
                len(y),
                1
            )
        )

        fit = sm.OLS(
            y,
            X
        ).fit(
            cov_type="HAC",
            cov_kwds={
                "maxlags": 3
            }
        )

        hac_rows.append({
            "Market_ID":
                market_id,
            "Model":
                model_name,
            "Specification":
                spec_name,
            "OOS_Events":
                len(g),
            "Mean_LogLoss_Gain":
                y.mean(),
            "HAC_Std_Error":
                fit.bse[0],
            "t_Statistic":
                fit.tvalues[0],
            "HAC_p_Value":
                fit.pvalues[0]
        })

hac_tests = pd.DataFrame(
    hac_rows
)

display(
    hac_tests.sort_values(
        [
            "HAC_p_Value",
            "Mean_LogLoss_Gain"
        ]
    )
)


## Step 22 — Compact RQ2 Summary for Felipe

The most useful feature should ideally:

1. improve CV Log Loss;
2. improve final Log Loss;
3. do so across several market-model combinations;
4. not rely on a single isolated test period.

This table summarizes that evidence.


In [ ]:
# ============================================================
# STEP 22: Compact RQ2 Summary
# ============================================================

rq2_summary = (
    feature_consistency[
        [
            "Specification",
            "Market_Model_Tests",
            "CV_Improved_Count",
            "Final_Improved_Count",
            "Both_Improved_Count",
            "Both_Improved_Share",
            "Mean_CV_LogLoss_Gain",
            "Mean_Final_LogLoss_Gain"
        ]
    ]
    .copy()
)

rq2_summary[
    "Overall_Interpretation"
] = np.where(
    (
        rq2_summary[
            "Both_Improved_Share"
        ] >= 0.50
    )
    &
    (
        rq2_summary[
            "Mean_CV_LogLoss_Gain"
        ] > 0
    )
    &
    (
        rq2_summary[
            "Mean_Final_LogLoss_Gain"
        ] > 0
    ),
    "Relatively consistent evidence of usefulness",
    np.where(
        (
            rq2_summary[
                "Both_Improved_Count"
            ] > 0
        ),
        "Some context-specific improvement",
        "No consistent improvement"
    )
)

display(
    rq2_summary.sort_values(
        [
            "Both_Improved_Count",
            "Mean_Final_LogLoss_Gain"
        ],
        ascending=[
            False,
            False
        ]
    )
)


## Step 23 — Save All Ablation Results


In [ ]:
# ============================================================
# STEP 23: Save Results Workbook
# ============================================================

OUTPUT_FILE = (
    "/content/"
    "Polymarket_Feature_Ablation_RQ2_All_Models_Results.xlsx"
)

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    cv_events.to_excel(
        writer,
        sheet_name="CV_Event_Predictions",
        index=False
    )

    final_events.to_excel(
        writer,
        sheet_name="Final_Event_Predictions",
        index=False
    )

    all_oos.to_excel(
        writer,
        sheet_name="All_OOS_Event_Predictions",
        index=False
    )

    cv_summary.to_excel(
        writer,
        sheet_name="CV_Summary",
        index=False
    )

    final_summary.to_excel(
        writer,
        sheet_name="Final_Summary",
        index=False
    )

    comparison.to_excel(
        writer,
        sheet_name="Baseline_Comparisons",
        index=False
    )

    feature_consistency.to_excel(
        writer,
        sheet_name="Feature_Consistency",
        index=False
    )

    market_feature_summary.to_excel(
        writer,
        sheet_name="Market_Feature_Summary",
        index=False
    )

    hac_tests.to_excel(
        writer,
        sheet_name="HAC_Tests",
        index=False
    )

    rq2_summary.to_excel(
        writer,
        sheet_name="Felipe_RQ2_Summary",
        index=False
    )

    feature_set_table.to_excel(
        writer,
        sheet_name="Feature_Set_Definitions",
        index=False
    )

print("Saved successfully:")
print(OUTPUT_FILE)


## Step 24 — Download the Results


In [ ]:
# ============================================================
# STEP 24: Download Results Workbook
# ============================================================

from google.colab import files

files.download(
    "/content/"
    "Polymarket_Feature_Ablation_RQ2_All_Models_Results.xlsx"
)
